# Experiment 5 — Lag sufficiency

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Compare feature families I (no lag), II (lag 1), III (lag 2)
under the Stage-2 winner's *model* class, to confirm the selected lag
order is supported.

In [ ]:
LAG_SPECS = ['I', 'II', 'III']      # lag 0, 1, 2 for raw-level families

rows = []
for p in POLICIES:
    _, mdl, alpha = SELECTED[p]
    if mdl not in ('OLS', 'Ridge'):
        mdl = 'Ridge'  # fall back to a linear model for lag comparison
    for feat in LAG_SPECS:
        meta, X, y, _ = nu.build_design_matrix(turns, p, feat)
        if len(X) < 50: continue
        pred, _ = nu.grouped_cv_predict(meta, X, y, mdl, n_splits=nu.OUTER_FOLDS)
        m = nu.cv_metrics_full(pred)
        rows.append({'policy': p, 'feature': feat,
                     'lag': nu.feature_spec(feat)['lag'],
                     'dim_phi': len(nu.feature_spec(feat)['cols']) + 1,
                     **m})
lag_tbl = pd.DataFrame(rows)
lag_tbl.pivot(index='policy', columns='feature', values='mae_norm').round(4)

In [ ]:
# One-SE rule per policy: what's the smallest lag within 1 SE of the best?
best_by_policy = (lag_tbl.sort_values(['policy','mae_norm'])
                        .groupby('policy', observed=True)
                        .head(1)[['policy','mae_norm','mae_norm_se']]
                        .rename(columns={'mae_norm':'best','mae_norm_se':'best_se'}))
joined = lag_tbl.merge(best_by_policy, on='policy')
joined['within_1se'] = joined['mae_norm'] <= joined['best'] + joined['best_se']

# Smallest lag within 1 SE
one_se_pick = (joined[joined['within_1se']]
                .sort_values(['policy','lag'])
                .groupby('policy', observed=True).head(1)
                [['policy','feature','lag','mae_norm']])
print('Smallest lag within 1 SE of best, per policy:')
print(one_se_pick.to_string(index=False))

### Figure: MAE vs lag order per policy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for p in POLICIES:
    sub = lag_tbl[lag_tbl['policy'] == p].sort_values('lag')
    ax.errorbar(sub['lag'], sub['mae_norm'], yerr=sub['mae_norm_se'],
                marker='o', lw=1.3, capsize=3, label=p)
ax.set_xlabel(r'lag order $\tau$')
ax.set_ylabel('normalized MAE')
ax.set_title('Lag sufficiency')
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
plt.savefig('figE5_lag_sufficiency.png', dpi=140, bbox_inches='tight')
plt.show()